<a href="https://colab.research.google.com/github/JuliPerezE/SRA-TP-Filtros-Grupo5/blob/main/notebooks/02_Sintesis_Circuital.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import sympy as sp
import numpy as np
from scipy import signal

# ==============================================================================
# 1. DEDUCCIÓN SIMBÓLICA CON SYMPY (ECUACIONES NODALES)
# ==============================================================================
print("=" * 70)
print(" 1. ANÁLISIS CIRCUTIAL SIMBÓLICO CON SYMPY")
print("=" * 70)

# Variables simbólicas de frecuencia y componentes
s = sp.Symbol('s')
R0, C0 = sp.symbols('R0 C0', positive=True)
R1, R2, C1, C2, K = sp.symbols('R1 R2 C1 C2 K', positive=True)
Vi, Vo1, Vo2, VA, VB, VX = sp.symbols('Vi Vo1 Vo2 VA VB VX')

# --- ETAPA 1: Celda Pasiva RC + Buffer (1.er Orden) ---
# Ecuación nodal en el nodo VX: (VX - Vi)/R0 + VX * s * C0 = 0
eq_rc = (VX - Vi)/R0 + VX * s * C0
sol_rc = sp.solve(eq_rc, VX)
H_c1_raw = sol_rc[0] / Vi

# Normalización para forma s + w_p1
num1, den1 = sp.fraction(H_c1_raw)
c1_s = den1.coeff(s, 1)
H_c1_norm = sp.simplify(num1 / c1_s) / sp.simplify(den1 / c1_s)

w_p1_sym = 1 / (R0 * C0)
K1_sym = 1

print("\n--- ETAPA 1: Pasabajos Pasivo RC (1.er Orden) ---")
print("• Ecuación nodal: (V_X - V_i)/R0 + V_X * s * C0 = 0")
print("• Función de transferencia H_c1(s):")
sp.pprint(H_c1_norm)
print(f"• Frecuencia de polo simbólica w_p1: {w_p1_sym}")
print(f"• Ganancia de la sección K1: {K1_sym}")

# --- ETAPA 2: Celda Activa Sallen-Key Pasabajos (2.º Orden) ---
# Ecuaciones nodales:
# Nodo A: (VA - Vo1)/R1 + (VA - VB)/R2 + (VA - Vo2)*s*C1 = 0
# Nodo B: (VB - VA)/R2 + VB*s*C2 = 0
# OpAmp ideal con ganancia K: Vo2 = K * VB  =>  VB = Vo2 / K
eq_SK1 = (VA - Vo1)/R1 + (VA - VB)/R2 + (VA - Vo2)*s*C1
eq_SK2 = (VB - VA)/R2 + VB*s*C2
eq_SK3 = VB - Vo2 / K

res_SK = sp.solve([eq_SK1, eq_SK2, eq_SK3], (VA, VB, Vo2))
H_c2_raw = sp.simplify(res_SK[Vo2] / Vo1)

# Normalización para forma canónica: s^2 + s*(w0/Q) + w0^2
num2, den2 = sp.fraction(sp.cancel(H_c2_raw))
den_poly = sp.Poly(den2, s)
c2_coeff = den_poly.all_coeffs()

num2_norm = sp.simplify(num2 / c2_coeff[0])
den2_norm = sp.simplify(den2 / c2_coeff[0])

a1_sym = den2_norm.coeff(s, 1)
a0_sym = den2_norm.coeff(s, 0)

w0_sym = sp.simplify(sp.sqrt(a0_sym))
Q_sym_gen = sp.simplify(w0_sym / a1_sym)
Q_sym_k1 = sp.simplify(Q_sym_gen.subs(K, 1))

print("\n--- ETAPA 2: Celda Sallen-Key Pasabajos (2.º Orden) ---")
print("• Ecuaciones nodales:")
print("  - Nodo A: (V_A - V_o1)/R1 + (V_A - V_B)/R2 + (V_A - V_o2)*s*C1 = 0")
print("  - Nodo B: (V_B - V_A)/R2 + V_B*s*C2 = 0")
print("  - OpAmp:  V_o2 = K * V_B")
print("\n• Función de transferencia H_c2(s) (Forma Canónica):")
sp.pprint(num2_norm / den2_norm)

print("\n• Parámetros simbólicos de la sección Bicuadrática:")
print("  - w0 (frecuencia natural):")
sp.pprint(w0_sym)
print("  - Q (factor de calidad general):")
sp.pprint(Q_sym_gen)
print("  - Q (para ganancia unitaria K = 1, buffer directo):")
sp.pprint(Q_sym_k1)
print("  - Ganancia K2 de la sección: K")


# ==============================================================================
# 2. COMPARACIÓN CON LA FUNCIÓN OBJETIVO Ht(s) (GRUPO 4: BESSEL N=3)
# ==============================================================================
print("\n" + "=" * 70)
print(" 2. COMPARACIÓN CON LA FUNCIÓN OBJETIVO Ht(s) (GRUPO 4)")
print("=" * 70)

# Especificaciones Grupo 4: Pasabajos f_p = 1000 Hz, Excepción Bessel N = 3
fp = 1000.0
wp_val = 2 * np.pi * fp

z, p, k = signal.bessel(N=3, Wn=wp_val, btype='lowpass', analog=True, output='zpk', norm='phase')

# Extraer polo real y par complejo conjugado
p_real = [p_i for p_i in p if np.abs(np.imag(p_i)) < 1e-5]
p_comp = [p_i for p_i in p if np.abs(np.imag(p_i)) >= 1e-5]

w_p1_num = np.abs(p_real)[0]
alpha = np.real(p_comp)[0]  # Parte real del polo complejo
beta = np.imag(p_comp)[0]   # Parte imaginaria

w0_num = np.sqrt(alpha**2 + beta**2)
Q_num = w0_num / (-2 * alpha)

print(f"\n📌 Función Objetivo Teórica Ht(s) para Pasabajos Bessel (N=3, fp={fp:.0f} Hz):")
print(f"   • Polos teóricos: {np.round(p, 2)}")
print(f"   • Ganancia k: {k:.2e}")

print("\n📌 Factorización en Secciones H1_t(s) y H2_t(s):")
print(f"   • Etapa 1 (1.er Orden):  w_p1 = {w_p1_num:.2f} rad/s ({w_p1_num/(2*np.pi):.2f} Hz) | K1 = 1")
print(f"     H1_t(s) = {w_p1_num:.2f} / (s + {w_p1_num:.2f})")
print(f"   • Etapa 2 (2.º Orden):  w0 = {w0_num:.2f} rad/s ({w0_num/(2*np.pi):.2f} Hz) | Q = {Q_num:.4f} | K2 = 1")
print(f"     H2_t(s) = {w0_num**2:.2e} / (s^2 + {w0_num/Q_num:.2f}*s + {w0_num**2:.2e})")

print("\n📌 Identificación de Parámetros Circuitales Hc(s) vs. Objetivo Ht(s):")
print(f"   1. Etapa 1:   w_p1 = 1 / (R0 * C0) = {w_p1_num:.2f} rad/s")
print(f"   2. Etapa 2:   w0   = 1 / sqrt(R1 * R2 * C1 * C2) = {w0_num:.2f} rad/s")
print(f"                 Q    = sqrt(C1 * R1 * R2) / (sqrt(C2) * (R1 + R2)) = {Q_num:.4f}  (para K=1)")
print("=" * 70)

 1. ANÁLISIS CIRCUTIAL SIMBÓLICO CON SYMPY

--- ETAPA 1: Pasabajos Pasivo RC (1.er Orden) ---
• Ecuación nodal: (V_X - V_i)/R0 + V_X * s * C0 = 0
• Función de transferencia H_c1(s):
        1        
─────────────────
      ⎛      1  ⎞
C₀⋅R₀⋅⎜s + ─────⎟
      ⎝    C₀⋅R₀⎠
• Frecuencia de polo simbólica w_p1: 1/(C0*R0)
• Ganancia de la sección K1: 1

--- ETAPA 2: Celda Sallen-Key Pasabajos (2.º Orden) ---
• Ecuaciones nodales:
  - Nodo A: (V_A - V_o1)/R1 + (V_A - V_B)/R2 + (V_A - V_o2)*s*C1 = 0
  - Nodo B: (V_B - V_A)/R2 + V_B*s*C2 = 0
  - OpAmp:  V_o2 = K * V_B

• Función de transferencia H_c2(s) (Forma Canónica):
                              K                               
──────────────────────────────────────────────────────────────
            ⎛ 2    K⋅s      s       s       s          1     ⎞
C₁⋅C₂⋅R₁⋅R₂⋅⎜s  - ───── + ───── + ───── + ───── + ───────────⎟
            ⎝     C₂⋅R₂   C₂⋅R₂   C₁⋅R₂   C₁⋅R₁   C₁⋅C₂⋅R₁⋅R₂⎠

• Parámetros simbólicos de la sección Bicuadrática:
  - w0 (fre